<a href="https://colab.research.google.com/github/ksenia-andreeva/kan-physics-recovery/blob/main/notebooks/kan_vs_mlp_sp1_noise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Практика: Восстановление физического закона из данных**

# Влияние уровня шума на восстановление закона (одна среда)
**Задача:** исследовать устойчивость KAN и MLP к зашумлению данных. Для одной среды k=4.0, c=0.3, m=1.0 варьируется уровень гауссова шума от 0% до 10%. Оценивается MSE и способность символьной регрессии выдавать компактную линейную формулу.

**Результат:** Численные коэффициенты α и β остаются близкими к истинным при всех шумах. Символьная регрессия "ломается" при шуме > 3%, порождая громоздкие экспоненциальные выражения, хотя численные предсказания точны.

# **Установка библиотек**

In [ ]:
!pip install git+https://github.com/KindXiaoming/pykan.git

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from scipy.integrate import solve_ivp
from kan import KAN
from kan.utils import ex_round

torch.set_default_dtype(torch.float32)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Работает на устройстве: {device}")

  Cloning https://github.com/KindXiaoming/pykan.git to /tmp/pip-req-build-0fhwvmxw
  Running command git clone --filter=blob:none --quiet https://github.com/KindXiaoming/pykan.git /tmp/pip-req-build-0fhwvmxw
  Resolved https://github.com/KindXiaoming/pykan.git to commit ecde4ec3274d3bef1ad737479cf126aed38ab530
  Preparing metadata (setup.py) ... done
  Created wheel for pykan: filename=pykan-0.2.8-py3-none-any.whl size=78235 sha256=533dcd5405f1bea0fa0f10b1fcc6cac3575e0525e1452ea859a4000572c0dfc6
  Stored in directory: /tmp/pip-ephem-wheel-cache-wty2a196/wheels/e5/c9/d6/a9b7aad8b3f7e1dde415462c7dd48df332ec616b149d51bcb8
Successfully built pykan
Работает на устройстве: cpu


# **Генерация данных и обучение KAN и MLP для одного уровня шума**

In [ ]:
def run_experiment(noise_level):
    # --- Генерация данных с шумом ---
    k, m, c = 4.0, 1.0, 0.3
    all_data = []

    def rhs(t, y):
        x, v = y
        a = -(k/m)*x - (c/m)*v
        return [v, a]

    for x0 in [1.0, 0.5, 2.0]:
        for v0 in [0.0, 0.5, -0.5]:
            sol = solve_ivp(rhs, (0, 20), y0=[x0, v0], t_eval=np.linspace(0, 20, 500))
            x, v = sol.y
            a_clean = -(k/m)*x - (c/m)*v
            a_std = np.std(a_clean)
            noise = np.random.normal(0, noise_level * a_std, size=len(a_clean))
            a_noisy = a_clean + noise

            for i in range(len(x)):
                all_data.append([x[i], v[i], a_noisy[i]])

    all_data = np.array(all_data)
    X = all_data[:, :2]
    y = all_data[:, 2]

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    X_train_t = torch.tensor(X_train.astype(np.float32))
    y_train_t = torch.tensor(y_train.astype(np.float32)).reshape(-1, 1)
    X_test_t  = torch.tensor(X_test.astype(np.float32))
    y_test_t  = torch.tensor(y_test.astype(np.float32)).reshape(-1, 1)

    dataset = {
        'train_input': X_train_t,
        'train_label': y_train_t,
        'test_input': X_test_t,
        'test_label': y_test_t
    }

    # --- Обучение KAN ---
    model_kan = KAN(width=[2,2,1], grid=10, k=1, seed=42, device=device)
    model_kan.fit(dataset, opt="LBFGS", steps=700, lamb=0.0)
    model_kan = model_kan.prune()
    model_kan.fit(dataset, opt="LBFGS", steps=100, lamb=0.0, update_grid=False)

    with torch.no_grad():
        pred_kan = model_kan(X_test_t)
        mse_kan = nn.functional.mse_loss(pred_kan, y_test_t).item()

    x_test = torch.tensor([[1.0, 0.0], [0.0, 1.0]], dtype=torch.float32)
    with torch.no_grad():
        a_pred = model_kan(x_test).numpy().flatten()
    alpha_kan, beta_kan = a_pred[0], a_pred[1]

    lib = ['x','x^2','x^3','x^4','exp','log','sqrt','tanh','sin','abs']
    model_kan.auto_symbolic(lib=lib)

    formula = ex_round(model_kan.symbolic_formula()[0][0],4)
    print("Символьная формула KAN:", formula)

    # --- Обучение MLP ---
    class MLP(nn.Module):
        def __init__(self):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(2, 16),
                nn.ReLU(),
                nn.Linear(16, 1))
        def forward(self, x):
            return self.net(x)

    model_mlp = MLP()
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model_mlp.parameters(), lr=0.001)

    X_tr, y_tr = dataset['train_input'], dataset['train_label']
    X_te, y_te = dataset['test_input'], dataset['test_label']

    for epoch in range(6000):
        model_mlp.train()
        optimizer.zero_grad()
        loss = criterion(model_mlp(X_tr), y_tr)
        loss.backward()
        optimizer.step()

    model_mlp.eval()
    with torch.no_grad():
        mse_mlp = criterion(model_mlp(X_te), y_te).item()

    result = {
        'noise': noise_level,
        'mse_kan': mse_kan,
        'mse_mlp': mse_mlp,
        'formula': str(formula),
        'alpha_kan': alpha_kan,
        'beta_kan': beta_kan,
        'true_alpha': -k/m,
        'true_beta': -c/m
    }

    return result

# **Цикл по разному гауссовому шуму и сбор результатов**

In [ ]:
noise_levels = [0.0, 0.01, 0.03, 0.05, 0.07, 0.1] # проверяемые уровни шума

results = []
for nl in noise_levels:
    print(f"Обработка шума {nl*100:.0f}%")
    res = run_experiment(nl)
    results.append(res)

Обработка шума 0%
checkpoint directory created: ./model
saving model version 0.0


| train_loss: 8.92e-04 | test_loss: 8.80e-04 | reg: 1.82e+01 | : 100%|█| 700/700 [02:54<00:00,  4.00


saving model version 0.1
saving model version 0.2


| train_loss: 1.26e-04 | test_loss: 2.35e-04 | reg: 1.47e+01 | : 100%|█| 100/100 [00:23<00:00,  4.25


saving model version 0.3
fixing (0,0,0) with 0
fixing (0,0,1) with exp, r2=1.000000238418579, c=2
fixing (0,1,0) with 0
fixing (0,1,1) with exp, r2=1.000000238418579, c=2
fixing (1,0,0) with 0
fixing (1,1,0) with exp, r2=1.000000238418579, c=2
saving model version 0.4


...

saving model version 0.3
fixing (0,0,0) with x, r2=0.9999989867210388, c=1
fixing (0,0,1) with exp, r2=1.000000238418579, c=2
fixing (0,1,0) with exp, r2=1.0000003576278687, c=2
fixing (0,1,1) with exp, r2=1.0000003576278687, c=2
fixing (1,0,0) with x, r2=0.9999898672103882, c=1
fixing (1,1,0) with exp, r2=1.0000003576278687, c=2
saving model version 0.4
Символьная формула KAN: -0.39*x_1 - 97.8607*exp(0.0071*exp(1.8081*x_1) + 0.0001*exp(3.4282*x_2)) + 98.5487 + 0.0135*exp(-5.8*x_2)
Обработка шума 10%
checkpoint directory created: ./model
saving model version 0.0


| train_loss: 1.55e-01 | test_loss: 1.47e-01 | reg: 1.64e+01 | : 100%|█| 700/700 [02:42<00:00,  4.30


saving model version 0.1
saving model version 0.2


| train_loss: 1.55e-01 | test_loss: 1.47e-01 | reg: 1.64e+01 | : 100%|█| 100/100 [00:11<00:00,  8.37


saving model version 0.3
fixing (0,0,0) with x, r2=0.9999844431877136, c=1
fixing (0,0,1) with exp, r2=1.0000003576278687, c=2
fixing (0,1,0) with exp, r2=1.000000238418579, c=2
fixing (0,1,1) with exp, r2=1.0000003576278687, c=2
fixing (1,0,0) with exp, r2=1.000000238418579, c=2
fixing (1,1,0) with exp, r2=1.000000238418579, c=2
saving model version 0.4
Символьная формула KAN: -0.0039*exp(4.4349*x_1 - 0.0671*exp(-4.9958*x_2)) - 19.6004*exp(0.0091*exp(1.0*x_2) - 0.2051*exp(-9.7246*x_1)) + 16.0623


# **Вывод таблицы по уровням шума**

In [ ]:
import pandas as pd

# сборка DataFrame
df = pd.DataFrame(results)
df['noise'] = df['noise'].apply(lambda x: f"{int(x*100)}%")

# Вывод таблицы с результатами
print("\n=== Таблица результатов ===")
print(df[[
    'noise',
    'mse_kan',
    'mse_mlp',
    'true_alpha',
    'alpha_kan',
    'true_beta',
    'beta_kan',
    'formula'
]].to_string(index=False))


=== Таблица результатов ===
noise      mse_kan  mse_mlp  true_alpha  alpha_kan  true_beta  beta_kan                                                                                                                  formula
   0% 5.545101e-08 0.000006        -4.0  -4.000097       -0.3 -0.300182                                                       31.555 - 35.5505*exp(0.0001*exp(4.2*x_2) - 0.1188*exp(-9.792*x_1))
   1% 2.280772e-04 0.000229        -4.0  -3.998261       -0.3 -0.303683                                       0.1184*x_1 - 25.3901*exp(0.0004*exp(3.4483*x_2) - 0.1756*exp(-9.939*x_1)) + 21.284
   3% 1.964519e-03 0.001997        -4.0  -4.011728       -0.3 -0.300145                                                      26.6992 - 26.5359*exp(0.0048*exp(3.4001*x_1) + 0.0021*exp(1.8*x_2))
   5% 5.605328e-03 0.005529        -4.0  -4.008934       -0.3 -0.298127                                            0.2291*x_1 - 54.2981*exp(0.0007*exp(2.2*x_2) - 0.081*exp(-8.4*x_1)) + 50.0978
   7% 